<a href="https://colab.research.google.com/github/fborrasumh/bioestadistica/blob/main/cuadernos/Tema_110_Integracion_de_datos_clinicos_y_tablas_de_frecuencias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# <font color='steelblue'>Tema 110. Integración de fuentes de datos clínicos y elaboración de tablas de frecuencias</font>

**Bioestadística (1964) · Grado en Medicina · 2.º curso · Curso 2027-2028**

**Módulo 3. Procesado inicial de datos**

**Autor:** Fernando Borrás Rocher · Universidad Miguel Hernández de Elche · ORCID [0000-0002-5519-4573](https://orcid.org/0000-0002-5519-4573)

**Duración:** 50 minutos de clase, más el estudio de los apartados de ampliación

**Práctica:** [unidad 110 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)

**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es). Todos los pacientes y datos de este cuaderno son sintéticos.

*Antes de trabajar, guarda tu copia: **Archivo → Guardar una copia en Drive**.*

## <font color='steelblue'>Configuración del cuaderno</font>

Ejecuta esta celda al empezar. Instala el módulo de preguntas interactivas y prepara el cuaderno.

In [ ]:
#@title Ejecutar configuración
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
print("Cuaderno configurado")

## <font color='steelblue'>Objetivos de aprendizaje</font>

Al terminar esta sesión serás capaz de:

1. Unir tablas de distintas fuentes por un identificador común y comprobar el resultado de la unión.
2. Elegir el tipo de unión adecuado (interna, por la izquierda o completa) según la pregunta clínica.
3. Resumir datos por grupos con `groupby`.
4. Construir tablas de frecuencias absolutas, relativas y acumuladas.
5. Agrupar una variable cuantitativa en categorías clínicas.
6. Construir e interpretar tablas de contingencia con porcentajes por filas o por columnas.

## <font color='steelblue'>Plan de la sesión</font>

| Bloque | Minutos |
| --- | --- |
| 1. Caso clínico de partida | 5 |
| 2. Conceptos clave | 10 |
| 3. Demostración: unir y contar | 20 |
| 4. Práctica en la app Bioestadística | 10 |
| 5. Autoevaluación | 5 |

## <font color='steelblue'>1. Caso clínico de partida (5 min)</font>

El registro cardiovascular del Tema 100 está limpio. Ahora la coordinadora quiere saber qué proporción de pacientes con diabetes tiene un **mal control glucémico** (HbA1c ≥ 8 %) en cada centro, y si los centros rurales difieren de los urbanos.

El problema es que la información está repartida en tres sitios:

- La **cohorte**, con los datos clínicos de cada paciente.
- Las **analíticas**, que envía el laboratorio, con la HbA1c, la creatinina y el LDL. No todos los pacientes tienen analítica, y el laboratorio ha enviado dos resultados de pacientes que no están en la cohorte.
- La tabla de **centros**, con el tipo de centro (urbano o rural) y su población.

Hay que **integrar** las tres fuentes sin perder ni duplicar pacientes, y después **contar**. Las tablas de frecuencias son el primer resultado de cualquier estudio clínico: la «Tabla 1» de los artículos.

## <font color='steelblue'>2. Conceptos clave (10 min)</font>

### <font color='steelblue'>2.1 Unir tablas</font>

Dos tablas se unen por una **clave**: una columna que identifica a cada paciente en ambas, como el código `id`. Con `pd.merge()` se elige qué filas conservar:

| Tipo (`how=`) | Conserva | Cuándo usarlo |
| --- | --- | --- |
| `"inner"` | Solo los pacientes presentes en las **dos** tablas | Análisis que necesita datos de ambas fuentes |
| `"left"` | **Todos** los de la tabla izquierda; los que faltan en la derecha quedan con `NaN` | Conservar la cohorte completa |
| `"outer"` | Todos los de **cualquiera** de las dos | Revisar discrepancias entre fuentes |

**Después de cada unión, comprueba el número de filas.** Si la clave está repetida en una de las tablas, la unión multiplica filas sin avisar.

### <font color='steelblue'>2.2 Frecuencias</font>

| Frecuencia | Definición | Ejemplo con 400 pacientes |
| --- | --- | --- |
| Absoluta (*n*) | Número de casos en cada categoría | 74 diabéticos |
| Relativa (*f*) | Proporción: *n* / total | 74 / 400 = 0,185 |
| Porcentual | Relativa × 100 | 18,5 % |
| Acumulada | Suma de las anteriores en variables **ordinales** | Pacientes con IMC hasta «sobrepeso» |

Las frecuencias acumuladas solo tienen sentido si las categorías tienen un orden.

### <font color='steelblue'>2.3 Tablas de contingencia</font>

Una **tabla de contingencia** cruza dos variables categóricas. Los porcentajes se pueden calcular sobre el total de cada **fila** o de cada **columna**, y responden a preguntas distintas:

- Porcentaje **por filas** (fila = tabaco): «De los fumadores, ¿qué porcentaje tuvo un evento?»
- Porcentaje **por columnas** (columna = evento): «De los que tuvieron un evento, ¿qué porcentaje fumaba?»

Confundirlos es uno de los errores más frecuentes al leer artículos.

## <font color='steelblue'>3. Demostración: unir y contar (20 min)</font>

### <font color='steelblue'>3.1 Cargar las tres fuentes</font>

In [ ]:
import pandas as pd

DATOS = "https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/datos/"
cohorte = pd.read_csv(DATOS + "cohorte_limpia.csv")
analiticas = pd.read_csv(DATOS + "analiticas.csv")
centros = pd.read_csv(DATOS + "centros.csv")

print("Cohorte:", cohorte.shape, "· Analíticas:", analiticas.shape, "· Centros:", centros.shape)
analiticas.head()

### <font color='steelblue'>3.2 Comprobar las claves antes de unir</font>

In [ ]:
print("¿id único en la cohorte?", cohorte["id"].is_unique)
print("¿id único en analíticas?", analiticas["id"].is_unique)

sin_analitica = set(cohorte["id"]) - set(analiticas["id"])
fuera_de_cohorte = set(analiticas["id"]) - set(cohorte["id"])
print("Pacientes de la cohorte sin analítica:", len(sin_analitica))
print("Analíticas de pacientes que no están en la cohorte:", sorted(fuera_de_cohorte))

Los conjuntos de Python (Tema 20) permiten restar listas de identificadores para encontrar las discrepancias.

### <font color='steelblue'>3.3 Tres tipos de unión</font>

In [ ]:
for tipo in ["inner", "left", "outer"]:
    unida = pd.merge(cohorte, analiticas, on="id", how=tipo)
    print(f"{tipo:>5}: {len(unida)} filas")

- `inner` conserva solo los 360 pacientes que están en ambas tablas.
- `left` conserva los 400 de la cohorte; los 40 sin analítica tendrán `NaN` en HbA1c.
- `outer` añade además las 2 analíticas huérfanas: útil para avisar al laboratorio, no para el análisis.

Para el estudio queremos la cohorte completa, así que usamos `left`. El parámetro `indicator=True` añade una columna que dice de dónde viene cada fila, y `validate="one_to_one"` detiene la unión si la clave estuviera repetida.

In [ ]:
datos = pd.merge(cohorte, analiticas, on="id", how="left", indicator=True, validate="one_to_one")
print(datos["_merge"].value_counts())
datos = datos.drop(columns="_merge")

datos = pd.merge(datos, centros, on="centro", how="left", validate="many_to_one")
print("Filas finales:", len(datos))
datos[["id", "centro", "tipo", "diabetes", "hba1c_pct"]].head()

### <font color='steelblue'>3.4 Resumir por grupos</font>

In [ ]:
datos.groupby("centro").agg(
    pacientes=("id", "count"),
    edad_media=("edad", "mean"),
    pas_media=("pas_mmHg", "mean"),
    eventos=("evento_cv_5a", "sum"),
).round(1)

### <font color='steelblue'>3.5 Tabla de frecuencias de una variable</font>

In [ ]:
n = datos["diabetes"].value_counts()
f = datos["diabetes"].value_counts(normalize=True)

pd.DataFrame({"n": n, "%": (f * 100).round(1)})

### <font color='steelblue'>3.6 De cuantitativa a categórica: la HbA1c</font>

`pd.cut()` agrupa una variable cuantitativa en intervalos. Con `right=False`, cada intervalo incluye su límite inferior y excluye el superior: [7, 8) significa «7 o más y menos de 8».

In [ ]:
diabeticos = datos[datos["diabetes"] == "Sí"].copy()

diabeticos["control"] = pd.cut(
    diabeticos["hba1c_pct"],
    bins=[0, 7, 8, 20],
    labels=["< 7 %", "7 a 7,9 %", "≥ 8 %"],
    right=False,
)

tabla = diabeticos["control"].value_counts().sort_index().to_frame("n")
tabla["%"] = (tabla["n"] / tabla["n"].sum() * 100).round(1)
tabla["% acumulado"] = tabla["%"].cumsum().round(1)
tabla

Los porcentajes se calculan sobre los diabéticos **con HbA1c disponible**. Siempre hay que indicar el denominador:

In [ ]:
print("Diabéticos:", len(diabeticos), "· con HbA1c:", diabeticos["hba1c_pct"].notna().sum())

### <font color='steelblue'>3.7 Tabla de contingencia</font>

In [ ]:
pd.crosstab(datos["fumador"], datos["evento_cv_5a"], margins=True, margins_name="Total")

Porcentajes **por filas**: de los fumadores y de los no fumadores, qué porcentaje tuvo un evento cardiovascular en cinco años.

In [ ]:
(pd.crosstab(datos["fumador"], datos["evento_cv_5a"], normalize="index") * 100).round(1)

Porcentajes **por columnas**: de los que tuvieron un evento y de los que no, qué porcentaje fumaba.

In [ ]:
(pd.crosstab(datos["fumador"], datos["evento_cv_5a"], normalize="columns") * 100).round(1)

Las dos tablas son correctas, pero responden a preguntas distintas. Para estudiar si el tabaco se asocia a más eventos, la pregunta natural es la primera: comparar el **riesgo** de evento entre fumadores y no fumadores. En el Tema 230 la cuantificaremos con el riesgo relativo.

### <font color='steelblue'>3.8 La pregunta del caso</font>

In [ ]:
diabeticos["mal_control"] = diabeticos["hba1c_pct"] >= 8

resumen = diabeticos.dropna(subset=["hba1c_pct"]).groupby("tipo")["mal_control"].agg(["sum", "count", "mean"])
resumen.columns = ["mal control (n)", "diabéticos con HbA1c (n)", "proporción"]
resumen["proporción"] = (resumen["proporción"] * 100).round(1).astype(str) + " %"
resumen

### <font color='steelblue'>Ampliación: unir filas de varios centros</font>

Si cada centro enviara su propio fichero con las mismas columnas, se apilarían con `pd.concat()`:

In [ ]:
norte = cohorte[cohorte["centro"] == "CS Norte"]
sur = cohorte[cohorte["centro"] == "CS Sur"]
apilado = pd.concat([norte, sur], ignore_index=True)
print(len(norte), "+", len(sur), "=", len(apilado))

### <font color='steelblue'>Ampliación: formato ancho y formato largo</font>

Las mediciones repetidas pueden guardarse en **formato ancho** (una columna por visita) o en **formato largo** (una fila por paciente y visita). Muchos análisis y gráficos necesitan el formato largo; `melt()` y `pivot()` convierten entre ambos.

In [ ]:
ancho = pd.DataFrame({"id": ["CV-0001", "CV-0002"], "pas_basal": [148, 136], "pas_6m": [138, 131], "pas_12m": [132, 128]})
largo = ancho.melt(id_vars="id", var_name="visita", value_name="pas_mmHg")
print(largo)
print(largo.pivot(index="id", columns="visita", values="pas_mmHg"))

### <font color='steelblue'>Ampliación: errores frecuentes</font>

**No comprobar el número de filas tras una unión.** Una clave repetida multiplica las filas; una clave mal escrita las pierde.

**Usar `inner` sin darse cuenta:** es el valor por defecto de `merge()` y elimina en silencio a los pacientes sin datos en la segunda tabla.

**Claves con formatos distintos:** `"CV-0001"` y `"cv-0001"`, o el número 1 y el texto `"0001"`, no coinciden.

**No indicar el denominador** de un porcentaje.

**Confundir porcentajes por filas y por columnas.**

**Calcular frecuencias acumuladas en variables nominales**, que no tienen orden.

## <font color='steelblue'>4. Práctica en la app Bioestadística (10 min)</font>

Abre la **[unidad 110 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)** y completa sus misiones:

1. **Misión básica (sin asistente):** predecir el número de filas que dará cada tipo de unión entre dos tablas pequeñas.
2. **Misión clínica (con pistas):** unir la cohorte con las analíticas y construir la tabla de frecuencias del filtrado glomerular por categorías.
3. **Misión reto (con asistente):** construir la «Tabla 1» de la cohorte por sexo con ayuda del asistente y verificar tres de sus cifras a mano.

Al terminar, exporta el **informe de la unidad**.

## <font color='steelblue'>5. Autoevaluación (5 min)</font>

Responde sin asistente y sin ejecutar código.

In [ ]:
#@title Pregunta interactiva
preguntas = [{'question': 'Una cohorte tiene 400 pacientes y la tabla de analíticas, 360 de ellos más 2 que no '
              'están en la cohorte. ¿Cuántas filas da merge(cohorte, analiticas, on="id", '
              'how="left")?',
  'type': 'multiple_choice',
  'answers': [{'answer': '400',
               'correct': True,
               'feedback': 'Correcto: se conservan todos los de la cohorte; los 40 sin analítica '
                           'quedan con NaN.'},
              {'answer': '360', 'correct': False, 'feedback': 'Eso daría how="inner".'},
              {'answer': '402', 'correct': False, 'feedback': 'Eso daría how="outer".'},
              {'answer': '362',
               'correct': False,
               'feedback': 'Esas son las filas de la tabla de analíticas.'}]},
 {'question': 'Tras unir dos tablas, el resultado tiene más filas que ambas. ¿Cuál es la causa más '
              'probable?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'La clave está repetida en alguna de las tablas',
               'correct': True,
               'feedback': 'Correcto: cada repetición multiplica filas. validate="one_to_one" lo '
                           'detecta.'},
              {'answer': 'Se ha usado how="left"',
               'correct': False,
               'feedback': 'left nunca da más filas que la tabla izquierda si la clave es única.'},
              {'answer': 'Hay valores perdidos',
               'correct': False,
               'feedback': 'Los NaN no multiplican filas.'},
              {'answer': 'La unión ha fallado',
               'correct': False,
               'feedback': 'La unión ha funcionado, pero con una clave no única.'}]},
 {'question': 'En una tabla de contingencia de tabaco (filas) por evento (columnas), ¿qué '
              'porcentaje responde a «de los fumadores, cuántos tuvieron un evento»?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'El porcentaje por filas',
               'correct': True,
               'feedback': 'Correcto: el denominador es el total de cada fila, es decir, de '
                           'fumadores.'},
              {'answer': 'El porcentaje por columnas',
               'correct': False,
               'feedback': 'Ese responde a «de los que tuvieron un evento, cuántos fumaban».'},
              {'answer': 'El porcentaje sobre el total de la tabla',
               'correct': False,
               'feedback': 'Ese da la proporción de fumadores con evento entre todos los '
                           'pacientes.'},
              {'answer': 'Ninguno: hace falta una prueba estadística',
               'correct': False,
               'feedback': 'La proporción se lee directamente de la tabla.'}]},
 {'question': '¿En qué variable tiene sentido calcular frecuencias acumuladas?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Categoría de control glucémico (< 7 %, 7 a 7,9 %, ≥ 8 %)',
               'correct': True,
               'feedback': 'Correcto: es ordinal, sus categorías tienen orden.'},
              {'answer': 'Centro de salud',
               'correct': False,
               'feedback': 'Es nominal: no tiene orden.'},
              {'answer': 'Grupo sanguíneo',
               'correct': False,
               'feedback': 'Es nominal: no tiene orden.'},
              {'answer': 'Sexo', 'correct': False, 'feedback': 'Es nominal y dicotómica.'}]},
 {'question': 'pd.cut(hba1c, bins=[0, 7, 8, 20], right=False). ¿En qué intervalo queda una HbA1c '
              'de 7,0 %?',
  'type': 'multiple_choice',
  'answers': [{'answer': '[7, 8)',
               'correct': True,
               'feedback': 'Correcto: con right=False, cada intervalo incluye su límite inferior.'},
              {'answer': '[0, 7)',
               'correct': False,
               'feedback': '7 no pertenece a [0, 7), que excluye el 7.'},
              {'answer': '[8, 20)', 'correct': False, 'feedback': '7 es menor que 8.'},
              {'answer': 'En ninguno',
               'correct': False,
               'feedback': 'Los intervalos cubren de 0 a 20.'}]}]
display_quiz(preguntas)

## <font color='steelblue'>Ideas clave</font>

- Las tablas se unen por una clave; se comprueba su unicidad antes y el número de filas después.
- El tipo de unión depende de la pregunta: `left` para conservar toda la cohorte.
- `groupby` resume por grupos; `value_counts` y `crosstab` construyen tablas de frecuencias y de contingencia.
- Todo porcentaje necesita su denominador; por filas y por columnas responden a preguntas distintas.

## <font color='steelblue'>Glosario</font>

| Término | Definición |
| --- | --- |
| Clave | Columna que identifica de forma única a cada unidad en una tabla |
| Unión (*merge*) | Combinación de dos tablas por una clave común |
| Frecuencia absoluta | Número de casos en una categoría |
| Frecuencia relativa | Proporción de casos en una categoría |
| Frecuencia acumulada | Suma de las frecuencias hasta una categoría, en variables ordinales |
| Tabla de contingencia | Tabla que cruza dos variables categóricas |
| Denominador | Total sobre el que se calcula una proporción |
| HbA1c | Hemoglobina glicosilada; refleja el control glucémico de los últimos meses |

## <font color='steelblue'>Para saber más</font>

- [Pandas: unir, combinar y concatenar](https://pandas.pydata.org/docs/user_guide/merging.html)
- [Pandas: agrupar datos con groupby](https://pandas.pydata.org/docs/user_guide/groupby.html)
- American Diabetes Association. Standards of Care in Diabetes. *Diabetes Care* (edición anual): objetivos de HbA1c.

---
*Tema 110 · Bioestadística (1964) · Grado en Medicina · Universidad Miguel Hernández de Elche · Fernando Borrás Rocher · CC BY 4.0*